# 03 Player Event Weights

This notebook converts player box scores into probabilities for who shoots, takes threes, assists, rebounds, and turns it over.

Important distinction: these weights are not the winner prediction model. They are player-level distributions used by the simulator so generated play-by-play and box scores feature realistic players from each team.


## Load Player Warehouse

Read the player-game fact table. This data powers player-level event assignment, while the winner/score model uses team-level rows from the previous notebook.


In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

# Resolve paths so the notebook works from either the repo root or notebooks/.
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
WAREHOUSE = ROOT / 'warehouse'
MODELS = ROOT / 'models'
MODELS.mkdir(exist_ok=True)

# Load the player-game warehouse. This is separate from the team-game model table.
players = pd.read_csv(WAREHOUSE / 'fact_player_game.csv')
players['GAME_DATE'] = pd.to_datetime(players['GAME_DATE'])
# The first three characters of MATCHUP identify the player's team for that row.
players['TEAM_ABBR'] = players['MATCHUP'].str[:3]
players.head()


,SEASON_ID,Player_ID,Game_ID,GAME_DATE,MATCHUP,WL,MIN,FGM,FGA,FG_PCT,...,TOV,PF,PTS,PLUS_MINUS,VIDEO_AVAILABLE,SEASON,SEASON_TYPE,PLAYER_ID,PLAYER_NAME,TEAM_ABBR
0,22015,203110.0,21500003.0,2015-10-27,GSW vs. NOP,W,29.0,3.0,12.0,0.250,...,1.0,5.0,10.0,20.0,1.0,2015-16,Regular Season,203110.0,Draymond Green,GSW
1,22015,201143.0,21500001.0,2015-10-27,ATL vs. DET,L,30.0,6.0,11.0,0.545,...,1.0,1.0,15.0,-5.0,1.0,2015-16,Regular Season,201143.0,Al Horford,ATL
2,22015,203084.0,21500003.0,2015-10-27,GSW vs. NOP,W,33.0,3.0,12.0,0.250,...,1.0,3.0,8.0,17.0,1.0,2015-16,Regular Season,203084.0,Harrison Barnes,GSW
3,22015,201959.0,21500002.0,2015-10-27,CHI vs. CLE,W,22.0,1.0,3.0,0.333,...,3.0,6.0,5.0,2.0,1.0,2015-16,Regular Season,201959.0,Taj Gibson,CHI
4,22015,201939.0,21500003.0,2015-10-27,GSW vs. NOP,W,36.0,14.0,26.0,0.538,...,2.0,1.0,40.0,12.0,1.0,2015-16,Regular Season,201939.0,Stephen Curry,GSW


## Convert Box Scores Into Event Weights

This section turns historical player production into probabilities. A player's share of team shots, threes, assists, rebounds, turnovers, and minutes becomes the simulator's sampling distribution.


In [2]:
# Use the latest available season so player weights represent current rosters.
season = players['SEASON'].max()
recent = players[(players['SEASON'] == season) & (players['MIN'].fillna(0) > 0)].copy()

# Aggregate each player's latest-season volume and per-game production.
agg = recent.groupby(['TEAM_ABBR', 'PLAYER_ID', 'PLAYER_NAME'], as_index=False).agg({
    'GAME_DATE': 'count',
    'MIN': ['sum', 'mean'],
    'FGM': ['sum', 'mean'],
    'FGA': ['sum', 'mean'],
    'FG3M': ['sum', 'mean'],
    'FG3A': ['sum', 'mean'],
    'FTM': ['sum', 'mean'],
    'FTA': ['sum', 'mean'],
    'AST': ['sum', 'mean'],
    'REB': ['sum', 'mean'],
    'TOV': ['sum', 'mean'],
    'PTS': ['sum', 'mean'],
})
agg.columns = ['_'.join(col).strip('_') for col in agg.columns]
agg = agg.rename(columns={
    'GAME_DATE_count': 'GAMES',
    'MIN_mean': 'MIN',
    'FGM_mean': 'FGM',
    'FGA_mean': 'FGA',
    'FG3M_mean': 'FG3M',
    'FG3A_mean': 'FG3A',
    'FTM_mean': 'FTM',
    'FTA_mean': 'FTA',
    'AST_mean': 'AST',
    'REB_mean': 'REB',
    'TOV_mean': 'TOV',
    'PTS_mean': 'PTS',
})

# Raw event columns estimate each player's role in different event types.
# These become within-team probability weights for the simulator.
agg['shot_raw'] = agg['FGA_sum'] + 0.44 * agg['FTA_sum'] + 0.25 * agg['AST_sum']
agg['scoring_raw'] = agg['shot_raw']
agg['three_raw'] = agg['FG3A_sum']
agg['free_throw_raw'] = agg['FTA_sum']
agg['assist_raw'] = agg['AST_sum']
agg['rebound_raw'] = agg['REB_sum']
agg['turnover_raw'] = agg['TOV_sum']
agg['minutes_raw'] = agg['MIN_sum']

agg['player_fg_pct'] = (agg['FGM_sum'] / agg['FGA_sum'].replace(0, np.nan)).fillna(0.45)
agg['player_fg3_pct'] = (agg['FG3M_sum'] / agg['FG3A_sum'].replace(0, np.nan)).fillna(0.35)
agg['player_ft_pct'] = (agg['FTM_sum'] / agg['FTA_sum'].replace(0, np.nan)).fillna(0.77)

raw_cols = ['shot_raw', 'scoring_raw', 'three_raw', 'free_throw_raw', 'assist_raw', 'rebound_raw', 'turnover_raw', 'minutes_raw']
# Normalize each event type within a team so probabilities sum to 1.
for col in raw_cols:
    weight_col = col.replace('_raw', '_weight')
    totals = agg.groupby('TEAM_ABBR')[col].transform('sum').replace(0, np.nan)
    agg[weight_col] = (agg[col] / totals).fillna(0)

# Build a rough rotation model from minutes so simulated box scores emphasize
# players who are actually likely to be on the floor.
agg = agg.sort_values(['TEAM_ABBR', 'MIN_sum'], ascending=[True, False])
agg['rotation_rank'] = agg.groupby('TEAM_ABBR').cumcount() + 1
agg['rotation_min_raw'] = np.where(agg['rotation_rank'] <= 10, agg['MIN'], 0.0)
agg['locked_star_min'] = np.where(agg['rotation_rank'] <= 2, agg['MIN'].clip(upper=36), 0.0)
locked_totals = agg.groupby('TEAM_ABBR')['locked_star_min'].transform('sum')
remaining_team_min = (240 - locked_totals).clip(lower=0)
bench_rotation_raw = np.where((agg['rotation_rank'] > 2) & (agg['rotation_rank'] <= 10), agg['MIN'], 0.0)
bench_totals = pd.Series(bench_rotation_raw, index=agg.index).groupby(agg['TEAM_ABBR']).transform('sum').replace(0, np.nan)
scaled_bench_min = bench_rotation_raw / bench_totals * remaining_team_min
agg['expected_min'] = np.where(agg['rotation_rank'] <= 2, agg['locked_star_min'], scaled_bench_min).round(1)
agg['expected_min'] = agg['expected_min'].fillna(0)
agg['availability_weight'] = (agg['expected_min'] / agg.groupby('TEAM_ABBR')['expected_min'].transform('max')).clip(lower=0.02)

# Re-tune scoring and free-throw weights toward active rotation players.
active_scoring = 0.60 * agg['PTS'] + 0.25 * (agg['FGA'] + 0.44 * agg['FTA']) + 0.15 * agg['expected_min']
agg['scoring_raw'] = active_scoring.clip(lower=0) * agg['availability_weight']
agg['free_throw_raw'] = agg['FTA'].clip(lower=0) * agg['availability_weight']
for col in ['scoring_raw', 'free_throw_raw']:
    weight_col = col.replace('_raw', '_weight')
    totals = agg.groupby('TEAM_ABBR')[col].transform('sum').replace(0, np.nan)
    agg[weight_col] = (agg[col] / totals).fillna(0)

weights = agg.sort_values(['TEAM_ABBR', 'minutes_weight'], ascending=[True, False])
weights.head(20)


,TEAM_ABBR,PLAYER_ID,PLAYER_NAME,GAMES,MIN_sum,MIN,FGM_sum,FGM,FGA_sum,FGA,...,free_throw_weight,assist_weight,rebound_weight,turnover_weight,minutes_weight,rotation_rank,rotation_min_raw,locked_star_min,expected_min,availability_weight
8,ATL,1629638.0,Nickeil Alexander-Walker,78,2603.0,33.371795,548.0,7.025641,1195.0,15.320513,...,0.206538,0.115789,0.075239,0.147810,0.131958,1,33.371795,33.371795,33.4,0.948864
12,ATL,1630552.0,Jalen Johnson,72,2535.0,35.208333,600.0,8.333333,1228.0,17.055556,...,0.293415,0.229150,0.207748,0.222628,0.128511,2,35.208333,35.208333,35.2,1.000000
14,ATL,1630700.0,Dyson Daniels,76,2523.0,33.197368,402.0,5.289474,777.0,10.223684,...,0.074905,0.181781,0.145985,0.123175,0.127902,3,33.197368,0.000000,29.7,0.843750
9,ATL,1630168.0,Onyeka Okongwu,74,2300.0,31.081081,413.0,5.581081,860.0,11.621622,...,0.119227,0.093522,0.157777,0.115876,0.116597,4,31.081081,0.000000,27.8,0.789773
19,ATL,1642258.0,Zaccharie Risacher,67,1500.0,22.388060,244.0,3.641791,536.0,8.000000,...,0.042209,0.030769,0.072150,0.052920,0.076042,5,22.388060,0.000000,20.0,0.568182
0,ATL,203468.0,CJ McCollum,41,1179.0,28.756098,285.0,6.951220,625.0,15.243902,...,0.132951,0.067611,0.035935,0.070255,0.059769,6,28.756098,0.000000,25.7,0.730114
18,ATL,1631243.0,Mouhamed Gueye,77,1178.0,15.298701,131.0,1.701299,290.0,3.766234,...,0.017331,0.027935,0.077204,0.030109,0.059718,7,15.298701,0.000000,13.7,0.389205
11,ATL,1630249.0,Vít Krejčí,46,1025.0,22.282609,143.0,3.108696,308.0,6.695652,...,0.024468,0.027530,0.027232,0.027372,0.051962,8,22.282609,0.000000,19.9,0.565341
3,ATL,1628379.0,Luke Kennard,46,945.0,20.543478,129.0,2.804348,240.0,5.217391,...,0.021996,0.038866,0.027793,0.029197,0.047906,9,20.543478,0.000000,18.4,0.522727
13,ATL,1630557.0,Corey Kispert,39,712.0,18.256410,125.0,3.205128,275.0,7.051282,...,0.041369,0.024291,0.024705,0.028285,0.036094,10,18.256410,0.000000,16.3,0.463068


## Inspect One Team

Before saving, inspect a team such as Phoenix to confirm that high-usage players receive larger scoring/shot weights and rotation players appear near the top.


In [3]:
# Inspect one team to make sure the weights look basketball-plausible.
team_abbr = 'PHX'
weights[weights['TEAM_ABBR'] == team_abbr][['PLAYER_NAME', 'MIN', 'PTS', 'shot_weight', 'three_weight', 'assist_weight', 'rebound_weight', 'turnover_weight']].head(12)


,PLAYER_NAME,MIN,PTS,shot_weight,three_weight,assist_weight,rebound_weight,turnover_weight
476,Collin Gillespie,28.475000,12.650000,0.113454,0.173469,0.184211,0.093375,0.113430
465,Royce O'Neale,28.371795,9.807692,0.080887,0.156062,0.103277,0.106225,0.093466
464,Devin Booker,33.562500,26.062500,0.177614,0.111044,0.191658,0.070817,0.182396
477,Oso Ighodaro,22.073171,6.475610,0.054070,0.000300,0.091857,0.119360,0.091652
466,Dillon Brooks,30.392857,20.196429,0.125067,0.110744,0.049156,0.057967,0.088929
473,Jordan Goodwin,22.514286,8.700000,0.071987,0.090636,0.074975,0.097373,0.061706
467,Grayson Allen,28.803922,16.490196,0.092074,0.136555,0.096822,0.044260,0.075318
474,Mark Williams,23.583333,11.666667,0.061739,0.000300,0.028798,0.136208,0.058984
478,Ryan Dunn,19.371429,5.814286,0.047593,0.050720,0.050645,0.084809,0.050817
471,Jalen Green,25.906250,17.750000,0.066814,0.068127,0.045184,0.033124,0.065336


## Save Player Weights

The simulator loads `models/player_event_weights.csv` to decide which player gets each generated event in the play-by-play and box score.


In [4]:
# Save the player-level event distributions consumed by the simulator.
out_path = MODELS / 'player_event_weights.csv'
weights.to_csv(out_path, index=False)
out_path


WindowsPath('c:/Users/jorda/Programs/nba_ml/models/player_event_weights.csv')